# Generative Adversarial Networks (GAN)

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

A refresher on the adversarial-game framework (Goodfellow et al., 2014) that
dominated generative modeling for ~7 years and still powers fast, sharp image
generation — even as diffusion models have taken the crown for most new work.

## 1. What & Why

A **GAN** trains two networks against each other:

- a **generator** `G` that maps a noise vector `z` to a fake sample `G(z)`, and
- a **discriminator** `D` that tries to tell real samples from `G`'s fakes.

They play a minimax game: `D` is trained to *catch* fakes, `G` is trained to
*fool* `D`. As `G` improves, `D`'s job gets harder, which pushes `G` harder — an
arms race that, at its (theoretical) equilibrium, leaves `G` producing samples
indistinguishable from the data and `D` reduced to guessing (output 0.5).

**The problem it solves.** Most generative models maximize a likelihood, which
forces an explicit, tractable density (and tends to produce blurry samples).
GANs are **likelihood-free**: they learn to *sample* from a complex
high-dimensional distribution directly, with no need to write down `p(x)`. That
implicit objective is what gives GANs their famously **sharp** outputs.

**Reach for a GAN when:**

- You want **fast, high-fidelity sampling** — one forward pass through `G` yields a
  sample (vs. dozens/hundreds of denoising steps for a diffusion model).
- The task is **image-to-image**: super-resolution, inpainting, style transfer,
  domain translation (Pix2Pix, CycleGAN), face synthesis (StyleGAN).
- You don't need a likelihood/density, just realistic samples.

**Don't reach for it when** you need **stable, predictable training**, a
**likelihood** or density estimate, guaranteed **mode coverage**, or simply the
**best image quality today** — modern **diffusion models** (see
`diffusion-models.ipynb`) usually beat GANs on fidelity and diversity and are far
less finicky to train.

## 2. Mental Model

A **counterfeiter vs. a detective**, locked in escalation:

```
        z ~ N(0, I)
           │
           ▼
     ┌───────────┐   fake G(z)
     │ Generator │ ───────────────┐
     │     G     │                │
     └───────────┘                ▼
                            ┌─────────────┐   D(x) ∈ (0,1)
   real x ~ p_data ───────▶ │Discriminator│ ───────────▶  "real" or "fake"?
                            │      D      │
                            └─────────────┘
                                   │
            ┌──────────────────────┴───────────────────────┐
            │  D's gradient says how to spot the fake       │
            │  G uses that SAME gradient, flipped, to       │
            │  make the next fake harder to spot.           │
            └───────────────────────────────────────────────┘
```

- The **counterfeiter (`G`)** never sees real money directly. It only learns from
  the **detective's feedback** — the gradient of `D` flowing back through `G(z)`.
- The **detective (`D`)** improves at spotting fakes, which forces better fakes.
- At the ideal **Nash equilibrium** the fakes are perfect, so the detective can do
  no better than a coin flip: `D(x) = 0.5` everywhere. The signal dries up, and
  training stops making progress. (In practice you stop well before this.)

## 3. Key Concepts

**The minimax value function** (Goodfellow et al., 2014):

$$
\min_G \max_D \; V(D,G) =
\mathbb{E}_{x \sim p_{\text{data}}}\!\big[\log D(x)\big]
+ \mathbb{E}_{z \sim p_z}\!\big[\log\big(1 - D(G(z))\big)\big]
$$

For a **fixed `G`**, the optimal discriminator is

$$
D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}.
$$

Plugging `D*` back in shows the generator is minimizing the **Jensen–Shannon
divergence** between `p_data` and `p_g`; the global optimum is `p_g = p_data`,
where `D*(x) = 1/2`.

| Term | What it is |
|---|---|
| **Generator `G`** | Maps latent noise `z` → sample; the model you keep at the end. |
| **Discriminator `D`** | Binary classifier real-vs-fake; a *learned, adaptive loss* for `G`. Discarded after training. |
| **Latent `z`** | Noise input (e.g. `N(0, I)`); walking `z` walks the output manifold. |
| **Non-saturating loss** | Train `G` to **maximize `log D(G(z))`** instead of minimize `log(1−D(G(z)))` — same fixed point, far stronger gradients early (see Example 2). |
| **Nash equilibrium** | Neither net can improve unilaterally; for GANs, `p_g = p_data`, `D = 0.5`. |
| **Mode collapse** | `G` maps many `z` to a few outputs — it fools `D` without covering the data. |
| **JS divergence** | What the original objective implicitly minimizes (the source of vanishing-gradient pain → motivates WGAN). |

**Key variants to recognize:** **DCGAN** (conv architecture + training recipe that
made GANs work), **WGAN / WGAN-GP** (Wasserstein/earth-mover distance + gradient
penalty for stability), **Conditional GAN** (condition `G` and `D` on a label),
**CycleGAN / Pix2Pix** (image translation), **StyleGAN** (state-of-the-art face
synthesis with a style-based generator).

## 4. Setup

The worked examples use **PyTorch** (CPU is fine — everything here is tiny). We
train a complete GAN on a 1-D toy distribution in seconds, then dissect the
gradient behaviour that motivates the non-saturating loss. An optional final cell
sketches a real image DCGAN and is gated behind an env var so the notebook always
runs top-to-bottom.

In [1]:
# %pip install torch numpy
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| numpy", np.__version__)

torch 2.12.1 | numpy 2.5.0


## 5. Worked Examples

### Example 1 — A complete GAN that learns a 1-D Gaussian

The smallest GAN that actually *works*: the real data is `N(4.0, 0.5)`, and `G`
must learn to turn standard noise into samples from that distribution — without
ever being told the mean or variance. We alternate one `D` step and one `G` step,
and watch the generated mean/std march toward the target while `D`'s loss settles
near `log 2 ≈ 0.69` × 2 (the equilibrium where it can no longer tell real from
fake).

In [2]:
REAL_MEAN, REAL_STD = 4.0, 0.5
Z_DIM, BATCH = 8, 256

def sample_real(n):
    return REAL_MEAN + REAL_STD * torch.randn(n, 1)

def noise(n):
    return torch.randn(n, Z_DIM)

G = nn.Sequential(
    nn.Linear(Z_DIM, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
D = nn.Sequential(
    nn.Linear(1, 32), nn.LeakyReLU(0.2),
    nn.Linear(32, 32), nn.LeakyReLU(0.2),
    nn.Linear(32, 1),                       # raw logit; BCEWithLogits adds the sigmoid
)

bce = nn.BCEWithLogitsLoss()
opt_d = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_g = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
real_label, fake_label = torch.ones(BATCH, 1), torch.zeros(BATCH, 1)

for step in range(4001):
    # --- train D: push real logits up, fake logits down ---
    real = sample_real(BATCH)
    fake = G(noise(BATCH)).detach()          # detach: no G gradient on the D step
    d_loss = bce(D(real), real_label) + bce(D(fake), fake_label)
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()

    # --- train G: non-saturating loss — make D call the fakes "real" ---
    gen = G(noise(BATCH))
    g_loss = bce(D(gen), real_label)
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()

    if step % 1000 == 0:
        with torch.no_grad():
            s = G(noise(4096))
        print(f"step {step:>4}  D_loss={d_loss.item():.3f}  G_loss={g_loss.item():.3f}  "
              f"gen mean={s.mean():.2f}  std={s.std():.2f}   (target {REAL_MEAN}/{REAL_STD})")

print("\nG learned to match the target moments without ever seeing them directly.")

step    0  D_loss=1.343  G_loss=0.634  gen mean=0.03  std=0.08   (target 4.0/0.5)


step 1000  D_loss=1.386  G_loss=0.686  gen mean=3.99  std=0.38   (target 4.0/0.5)


step 2000  D_loss=1.386  G_loss=0.680  gen mean=3.92  std=0.39   (target 4.0/0.5)


step 3000  D_loss=1.386  G_loss=0.715  gen mean=4.01  std=0.42   (target 4.0/0.5)


step 4000  D_loss=1.386  G_loss=0.702  gen mean=4.10  std=0.45   (target 4.0/0.5)

G learned to match the target moments without ever seeing them directly.


### Example 2 — Why the non-saturating generator loss

The original paper proposes `G` minimize `log(1 − D(G(z)))`, but immediately
recommends *instead* maximizing `log D(G(z))`. Reason: early in training `D`
easily rejects fakes, so `D(G(z))` is near 0 — exactly where the **saturating**
loss has a vanishing gradient and `G` barely learns. The **non-saturating** loss
has its *largest* gradient there. We make this concrete by computing each loss's
gradient with respect to `D(G(z))` across the range.

In [3]:
p = np.linspace(0.01, 0.99, 99)        # D's predicted P(fake is real)

# Saturating generator loss  L = log(1 - p)        -> dL/dp = -1 / (1 - p)
grad_saturating = -1.0 / (1.0 - p)
# Non-saturating generator loss L = -log(p)        -> dL/dp = -1 / p
grad_nonsat = -1.0 / p

print(f"{'D(G(z))':>8} | {'|grad| saturating':>18} | {'|grad| non-saturating':>21}")
print("-" * 54)
for pv in (0.05, 0.20, 0.50, 0.90):
    i = int(np.argmin(np.abs(p - pv)))
    print(f"{pv:>8.2f} | {abs(grad_saturating[i]):>18.2f} | {abs(grad_nonsat[i]):>21.2f}")

print("\nWhen the generator is losing (D(G(z))≈0.05) the non-saturating loss gives a")
print("~20x stronger learning signal — which is why it is the default in practice.")

 D(G(z)) |  |grad| saturating | |grad| non-saturating
------------------------------------------------------
    0.05 |               1.05 |                 20.00
    0.20 |               1.25 |                  5.00
    0.50 |               2.00 |                  2.00
    0.90 |              10.00 |                  1.11

When the generator is losing (D(G(z))≈0.05) the non-saturating loss gives a
~20x stronger learning signal — which is why it is the default in practice.


### Example 3 — A real image DCGAN (optional, gated)

Convincing image GANs (DCGAN, StyleGAN) need a GPU and a real dataset, so this
cell only runs when you opt in. It shows the *shape* of the idea: a transposed-conv
generator upsampling noise to an image, and a strided-conv discriminator scoring
it. The notebook still executes top-to-bottom without it.

In [4]:
import os

if os.getenv("RUN_DCGAN"):
    # Requires: pip install torchvision ; downloads MNIST (~12 MB); slow on CPU.
    import torch.nn as nn

    nz, ngf, ndf = 100, 32, 32
    netG = nn.Sequential(
        nn.ConvTranspose2d(nz, ngf * 2, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
        nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),
        nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False), nn.Tanh(),          # -> 1x28x28
    )
    netD = nn.Sequential(
        nn.Conv2d(1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
        nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
        nn.Conv2d(ndf * 2, 1, 7, 1, 0, bias=False), nn.Flatten(),            # -> logit
    )
    z = torch.randn(4, nz, 1, 1)
    print("G(z) image batch shape:", tuple(netG(z).shape))
    print("D(image) logit shape  :", tuple(netD(netG(z)).shape))
    print("Now wire up the same alternating loop as Example 1 over real MNIST batches.")
else:
    print("Set RUN_DCGAN=1 (and `pip install torchvision`) to build a DCGAN on MNIST.")
    print("Shape of the idea:")
    print("  G: z(100) --ConvTranspose stack--> 1x28x28 image  (Tanh output, BatchNorm, ReLU)")
    print("  D: 1x28x28 image --Conv stack--> logit            (LeakyReLU, no pooling)")
    print("  Loss/loop: identical alternating D-step / G-step from Example 1.")

Set RUN_DCGAN=1 (and `pip install torchvision`) to build a DCGAN on MNIST.
Shape of the idea:
  G: z(100) --ConvTranspose stack--> 1x28x28 image  (Tanh output, BatchNorm, ReLU)
  D: 1x28x28 image --Conv stack--> logit            (LeakyReLU, no pooling)
  Loss/loop: identical alternating D-step / G-step from Example 1.


## 6. Gotchas & Pitfalls

- **Use the non-saturating loss.** Train `G` to maximize `log D(G(z))` (i.e.
  `bce(D(fake), real_label)`), never minimize `log(1 − D(G(z)))` — see Example 2.
- **Mode collapse.** `G` finds a few outputs that reliably fool `D` and stops
  covering the rest of the data. Symptoms: low diversity, repeated samples.
  Mitigations: minibatch discrimination, unrolled GANs, WGAN-GP, more `z` variety.
- **Don't trust the loss curves.** GAN losses oscillate and do **not** monotonically
  decrease — a "low G loss" can mean `D` is broken, not that samples are good.
  Evaluate samples with **FID / Inception Score**, not the training loss.
- **Keep `D` and `G` balanced.** If `D` becomes near-perfect, `D(G(z))→0`, the
  gradient to `G` vanishes, and training stalls. Tactics: matched capacities,
  **two-time-scale update rule (TTUR)** (different lrs), label smoothing, or `D`
  regularization (spectral norm / gradient penalty).
- **`detach()` the fake on the D step.** Forgetting it backprops into `G` while
  you're updating `D` — a subtle, common bug.
- **Architecture hygiene (DCGAN rules).** Strided convs instead of pooling,
  **BatchNorm** in `G` and `D` (but *not* the `G` output / `D` input layers),
  **LeakyReLU** in `D`, `Tanh` output on `G` with inputs scaled to `[-1, 1]`.
- **Adam `betas=(0.5, 0.999)`.** The default `0.9` momentum is too high and
  destabilizes the adversarial dynamics; `0.5` is the standard GAN setting.
- **WGAN changes the rules.** With Wasserstein loss the critic outputs a raw score
  (no sigmoid/BCE), you don't use BatchNorm in the critic with gradient penalty,
  and you can actually read the loss as a quality proxy.

## 7. When to Use vs Alternatives

| Option | Pick it when… | Trade-off vs GAN |
|---|---|---|
| **Diffusion models** | You want best-in-class image fidelity & diversity and stable training. | Far more stable, better mode coverage, but **slow sampling** (many steps). See `diffusion-models.ipynb`. |
| **VAE** | You need a smooth latent space, an encoder, and a likelihood bound. | Stable & principled, gives `p(x)` bound + encoder, but samples are **blurrier**. See `vae.ipynb`. |
| **Normalizing flows** | You need **exact** likelihoods and invertible mapping. | Exact density + invertibility, but architecturally constrained and heavier. See `normalizing-flows.ipynb`. |
| **Autoregressive (PixelCNN, Transformers)** | Exact likelihood and strong on discrete data/text. | Exact `p(x)`, great quality, but **slow** sequential sampling. |
| **WGAN-GP / StyleGAN (still a GAN)** | You want GAN speed but more stability/quality. | More stable training and SOTA faces, but more moving parts. |

**Rule of thumb (2026):** for **new** generative-image work, reach for a
**diffusion model** first — it's more stable and usually higher quality. Choose a
**GAN** when **single-pass sampling speed** matters (real-time / interactive
generation), for **image-to-image translation** (CycleGAN/Pix2Pix), or for
high-quality **face synthesis** (StyleGAN). Choose a **VAE** when you want a
well-behaved latent space and an encoder. Closely related notebooks: `vae.ipynb`,
`diffusion-models.ipynb`, `normalizing-flows.ipynb`.

## 8. Resources

- **Generative Adversarial Nets** — Goodfellow et al., 2014 (the original paper):
  https://arxiv.org/abs/1406.2661
- **NIPS 2016 Tutorial: GANs** — Goodfellow's tutorial, the best single overview of
  the theory and the practical failure modes: https://arxiv.org/abs/1701.00160
- **DCGAN** — Radford et al., 2015, the conv architecture + recipe that made GANs
  trainable: https://arxiv.org/abs/1511.06434
- **WGAN / WGAN-GP** — Arjovsky et al. (https://arxiv.org/abs/1701.07875) and
  Gulrajani et al. (https://arxiv.org/abs/1704.00028) on stabilizing training.
- **From GAN to WGAN** — Lilian Weng's deep-dive on the math and divergences:
  https://lilianweng.github.io/posts/2017-08-20-gan/
- **PyTorch DCGAN tutorial** — runnable end-to-end image GAN:
  https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html